# 学习地图

> **本章主题：在 Python 中，函数是"一等对象（First-Class Object）"，和整数、字符串一样，是可以被自由摆弄的普通对象。**

编程语言研究者给"一等对象"下的定义是——一个程序实体若能同时满足下面四点，就是一等对象：

| 一等对象的四个特征 | 含义 |
|---|---|
| 在运行时创建 | 函数不是编译期写死的符号，而是运行中真正造出来的对象 |
| 能赋值给变量或存入数据结构 | 可以把函数塞进列表、字典、字段 |
| 能作为参数传给函数 | `map`、`sorted(key=...)` 都靠这个 |
| 能作为函数的返回结果 | 工厂函数、装饰器的基础 |

整数、字符串、字典早就是一等对象了——所以"一等函数"其实没什么特别的：**在 Python 中，所有函数都是一等函数**。

本章按"从观察到应用"的脉络展开：

| 问题 | 对应小节 |
|---|---|
| 函数真的是对象吗？ | 7.2 将函数视为对象 |
| 什么样的函数算"高级玩法"？ | 7.3 高阶函数 |
| 一次性小函数怎么写得干净？ | 7.4 匿名函数 |
| 除了函数，还有谁能被括号调用？ | 7.5–7.6 可调用对象 |
| 怎样把参数声明写得又灵活又严谨？ | 7.7 参数声明（`*` 与 `/`） |
| 标准库给了哪些函数式工具？ | 7.8 operator 与 functools |

> 本章和第三部分的大部分内容，都在探讨"把函数当成对象"带来的实际用法。

# 7.1 本章新增内容

相对第 1 版，本章的主要变化：

- "7 种可调用对象" → **"9 种可调用对象"**：新增了**原生协程函数**（Python 3.5）与**异步生成器函数**（Python 3.6），它们与生成器一起在第 21 章介绍；
- 新增 **"7.7.1 仅限位置参数"** 一节，涵盖 Python 3.8 的新语法（`/`）；
- 把"运行时访问函数注解"的讨论移到了 **15.5.1**，因为注解现在归"类型提示"管。

# 7.2 将函数视为对象

Python 函数是货真价实的对象。下面这个控制台会话创建函数、调用它、读取它的 `__doc__` 属性，并确认函数本身就是 `function` 类的实例。

In [1]:
# 示例 7.1：创建一个函数，并检查函数类型
def factorial(n):
    '''returns n!'''
    return 1 if n < 2 else n * factorial(n - 1)

print(factorial(42))
print(factorial.__doc__)      # __doc__ 是函数对象的属性之一
print(type(factorial))        # factorial 是 function 类的实例

1405006117752879898543142606244511569936384000000000
returns n!
<class 'function'>


`__doc__` 属性用于生成对象的帮助文本——在控制台里执行 `help(factorial)`，Python 会读取 `factorial.__doc__` 来显示。

函数对象的"一等"本性还体现在：**可以赋给别的名字，也可以当参数传出去。**

In [2]:
# 示例 7.2：通过其他名称调用 factorial，并把 factorial 作为参数传递
fact = factorial                 # 把 factorial 绑定到另一个名字
print(fact)
print(fact(5))
print(map(factorial, range(11))) # map 返回的是一个可迭代对象（惰性求值）
print(list(map(factorial, range(11))))

<function factorial at 0x000002542D5D3EC0>
120
[1, 1, 2, 6, 24, 120, 720, 5040, 40320, 362880, 3628800]


> **记忆要点：`def` 语句在运行时执行，效果是"创建一个 function 对象并绑定到某个名字"。** 名字只是标签（第 6 章的主题），同一个函数对象可以有多个名字。

有了"一等函数"，我们就能用函数式风格编程，其中最关键的工具是**高阶函数**。

# 7.3 高阶函数

**接收函数为参数、或把函数作为结果返回的函数，就是高阶函数（Higher-Order Function）。**

`map` 是一个例子；内置函数 `sorted` 也是——它的可选参数 `key` 接收一个函数，`sorted` 会把该函数应用到每一项上，按返回值排序。想按单词长度排序，只需把 `len` 传给 `key`。

In [3]:
# 示例 7.3：根据单词长度排序列表
fruits = ['strawberry', 'fig', 'apple', 'cherry', 'raspberry', 'banana']
print(sorted(fruits, key=len))

['fig', 'apple', 'cherry', 'banana', 'raspberry', 'strawberry']


任何**单参数函数**都可以作为 `key` 的值。例如，做韵律词典时可能要按单词的**反向拼写**排序——后缀为 `berry` 的单词就能排在一起。

In [4]:
# 示例 7.4：按反向拼写对单词列表排序
def reverse(word):
    return word[::-1]

print(reverse('testing'))
print(sorted(fruits, key=reverse))

gnitset
['banana', 'apple', 'fig', 'raspberry', 'strawberry', 'cherry']


> 注意：排序并没有改动列表里的单词，只是把它们**反向拼写**当作排序标准。`sorted` 是**非就地**排序，返回新列表。

函数式编程里最著名的高阶函数有 `map`、`filter`、`reduce`、`apply`。`apply` 在 Python 2.3 被弃用、Python 3 被删除——现在已经不需要它了，直接写 `fn(*args, **kwargs)` 即可。

`map`、`filter`、`reduce` 仍然存在，但**大多数场景都有更好的替代品**。

## 7.3.1 map、filter 与 reduce 的替代品

自从有了**列表推导式（Listcomp）**与**生成器表达式（Genexp）**，`map` 和 `filter` 就不那么重要了——推导式更简洁、更易读。

In [5]:
# 示例 7.5：列表推导式与高阶函数比较
print(list(map(factorial, range(6))))                                  # 0! ~ 5!
print([factorial(n) for n in range(6)])                                # 等价的列表推导式
print(list(map(factorial, filter(lambda n: n % 2, range(6)))))         # map + filter + lambda
print([factorial(n) for n in range(6) if n % 2])                       # 等价的推导式，无需 lambda

[1, 1, 2, 6, 24, 120]
[1, 1, 2, 6, 24, 120]
[1, 6, 120]
[1, 6, 120]


> **关键对照：**
> - Python 3 中 `map` / `filter` 返回**生成器**（迭代器），所以它们最贴近的替代品是**生成器表达式**；
> - Python 2 中它们返回列表，最贴近的替代品才是**列表推导式**。

`reduce` 在 Python 3 里从内置函数"降级"到了 `functools` 模块。它最常用于**求和**，但自从 2003 年 `sum` 出现后，求和有了更好、更快、更易读的写法。

In [6]:
# 示例 7.6：分别用 reduce 与 sum 计算 0-99 之间的整数和
from functools import reduce
from operator import add
print(reduce(add, range(100)))
print(sum(range(100)))

4950
4950


`sum` 与 `reduce` 的**规约（Reducing）**思路一致：把某个操作连续应用到序列的每个项上，累计前一个结果，最终把一串值规约成一个值。

内置的规约函数还有 `all` 与 `any`：

In [7]:
print('all([])        ->', all([]))         # 空可迭代对象 → True
print('all([1, 2, 3]) ->', all([1, 2, 3]))  # 全为真 → True
print('all([1, 0, 3]) ->', all([1, 0, 3]))  # 有假 → False
print('any([])        ->', any([]))         # 空可迭代对象 → False
print('any([0, 0, 3]) ->', any([0, 0, 3]))  # 有真 → True
print('any([0, 0, 0]) ->', any([0, 0, 0]))  # 全为假 → False

all([])        -> True
all([1, 2, 3]) -> True
all([1, 0, 3]) -> False
any([])        -> False
any([0, 0, 3]) -> True
any([0, 0, 0]) -> False


> **记忆要点：`all` 空为真、`any` 空为假。** 这一对是"全称/存在"量词的实现：`all` = "所有都满足"，`any` = "至少有一个满足"。

为了配合高阶函数，有时需要临时造一个**一次性小函数**——这正是匿名函数登场的原因。

# 7.4 匿名函数

`lambda` 关键字创建匿名函数。受 Python 语法限制，**lambda 的主体必须是纯表达式**：

- 不能包含语句（如 `while`、`try`）；
- 赋值（带 `=`）是语句，也不能出现在 lambda 里；
- 新型赋值表达式 `:=` 虽然能用，但会让 lambda 过于复杂，应改用 `def` 重构。

匿名函数最适合放在**高阶函数的参数列表**里。用 `lambda` 改写示例 7.4 的单词排序：

In [8]:
# 示例 7.7：用 lambda 表达式改写示例 7.4
print(sorted(fruits, key=lambda word: word[::-1]))

['banana', 'apple', 'fig', 'raspberry', 'strawberry', 'cherry']


除了给高阶函数当参数，Python 里很少使用匿名函数。**若 lambda 难以阅读，就该重构。** Fredrik Lundh 的四步重构法：

1. 写一句注释，说明这个 lambda 的作用；
2. 研究这句注释，想一个能概括它的**名字**；
3. 用 `def` 语句，以该名字把 lambda 改写成常规函数；
4. 删掉那行注释。

> lambda 只是**语法糖**——它创建的就是普通的函数对象，和使用 `def` 定义毫无二致。它只是 Python 众多**可调用对象**中的一种。

# 7.5 9 种可调用对象

除了函数，调用运算符 `()` 还能用在别的对象上。判断对象是否可调用，用内置的 `callable()`。自 Python 3.6 起，数据模型列出了 **9 种可调用类型**：

| 可调用类型 | 说明 |
|---|---|
| 用户定义的函数 | 用 `def` 或 `lambda` 创建 |
| 内置函数 | 用 C 语言（CPython）实现，如 `len`、`time.strftime` |
| 内置方法 | 用 C 实现的方法，如 `dict.get` |
| 方法 | 在类主体中定义的函数 |
| 类 | 调用类会先跑 `__new__` 造实例，再跑 `__init__` 初始化，最后返回实例——所以"调用类"就像"调用函数" |
| 类的实例 | 若类定义了 `__call__`，其**实例**也能像函数一样被调用 |
| 生成器函数 | 函数体含 `yield`，被调用时返回生成器对象 |
| 原生协程函数（≥ 3.5） | 用 `async def` 定义，被调用时返回协程对象 |
| 异步生成器函数（≥ 3.6） | 用 `async def` 定义且函数体含 `yield`，配 `async for` 使用 |

后三种与其他类型不同：它们的返回值不是应用数据，而是**需要进一步处理才能产生数据、或执行某种操作的对象**。

In [9]:
# 判断对象是否可调用
print(abs, str, 'Ni!')
print([callable(obj) for obj in (abs, str, 'Ni!')])

<built-in function abs> <class 'str'> Ni!
[True, True, False]


> **记忆要点：鉴于可调用类型繁多，判断对象是否可调用最安全的方法就是 `callable()`。** 别靠 `isinstance` 或猜。

下面看看如何把**类的实例**也变成可调用对象。

# 7.6 用户定义的可调用类型

不止函数是对象——**任何 Python 对象**，只要实现了 `__call__` 实例方法，就能像函数一样被调用。

示例 7.8 的 `BingoCage` 接受一个可迭代对象，在内部保存一份**乱序**的元素列表；调用实例时，从列表里弹出一个元素。

In [10]:
# 示例 7.8：bingocall.py —— 从乱序的列表中取出一个元素
import random

class BingoCage:

    def __init__(self, items):
        self._items = list(items)      # 建本地副本，防止对外部列表产生副作用
        random.shuffle(self._items)    # 打乱顺序

    def pick(self):
        try:
            return self._items.pop()
        except IndexError:
            raise LookupError('pick from empty BingoCage')

    def __call__(self):                # 让 bingo() 成为 bingo.pick() 的快捷方式
        return self.pick()

In [11]:
# 演示：把实例当作函数调用
bingo = BingoCage(range(3))
print(bingo.pick())
print(bingo())
print(callable(bingo))

1
2
True


> **为什么要自己写 `BingoCage`，而不是用 `random.choice`？** 因为 `random.choice` 可能多次返回同一个元素（选中项不会被移除）；而只要实例里的值唯一，`BingoCage` 就**永远不会返回重复结果**。

实现 `__call__` 的类，适合创建"带内部状态的函数式对象"（比如剩余项）。`__call__` 的另一个好用例是实现**装饰器**——装饰器必须可调用，且常需要在多次调用之间"记住"东西（如记忆化 memoization）。

> 函数式编程里，要创建"保持内部状态的函数"，用的是**闭包**——见第 9 章。

接下来探讨 Python 声明形参与传入实参的强大语法。

# 7.7 从位置参数到仅限关键字参数

Python 函数的参数处理机制极其灵活。与之密切相关的是：调用函数时，可用 `*` 和 `**` 把可迭代对象、映射**解包**成单独的参数。

示例 7.9 的 `tag` 函数用于生成 HTML 标签，它把几种参数形式都用上了：

- `name`：普通位置参数；
- `*content`：捕获任意数量的**位置参数**（变量参数），存入一个 tuple；
- `class_`：**仅限关键字参数**（因为它排在 `*content` 之后）；
- `**attrs`：捕获其余**关键字参数**，存入一个 dict。

In [12]:
# 示例 7.9：函数 tag 用于生成 HTML 标签
def tag(name, *content, class_=None, **attrs):
    '''生成一个或多个 HTML 标签'''
    if class_ is not None:
        attrs['class'] = class_
    attr_pairs = (f' {attr}="{value}"' for attr, value in sorted(attrs.items()))
    attr_str = ''.join(attr_pairs)
    if content:
        elements = (f'<{name}{attr_str}>{c}</{name}>' for c in content)
        return '\n'.join(elements)
    else:
        return f'<{name}{attr_str} />'

In [13]:
# 示例 7.10：tag 函数的多种调用方式
print(tag('br'))                                          # 空标签
print(tag('p', 'hello'))                                  # *content 捕获 ('hello',)
print(tag('p', 'hello', 'world'))                         # 多个内容 → 多个标签
print(tag('p', 'hello', id=33))                           # 未命名的关键字参数进 **attrs
print(tag('p', 'hello', 'world', class_='sidebar'))       # class_ 只能用关键字传入
print(tag(content='testing', name='img'))                 # 第一个位置参数也能按关键字传
my_tag = {'name': 'img', 'title': 'Sunset Boulevard',
          'src': 'sunset.jpg', 'class': 'framed'}
print(tag(**my_tag))                                      # ** 把 dict 展开成独立的关键字实参

<br />
<p>hello</p>
<p>hello</p>
<p>world</p>
<p id="33">hello</p>
<p class="sidebar">hello</p>
<p class="sidebar">world</p>
<img content="testing" />
<img class="framed" src="sunset.jpg" title="Sunset Boulevard" />


> **记忆要点：**
> - `*content` 与 `**attrs` 分别把"多余的位置参数"和"多余的关键字参数"收进 tuple / dict；
> - 同名形参会优先绑定：`my_tag` 里的 `name` 绑定到 `name`，其余进 `**attrs`；
> - dict 里可以用 `'class'` 作键，因为它是**字符串**，和保留字 `class` 不冲突（这也正是函数形参要叫 `class_` 的原因）。

**仅限关键字参数（Keyword-only Argument）** 是 Python 3 的新功能：把参数放在 `*` **之后**，它就绝不能按位置传入。若不需要可变位置参数、但仍要仅限关键字参数，就单写一个 `*`。

In [14]:
# 示例 7.11：* 后面的参数 b 是仅限关键字参数
def f(a, *, b):
    return a, b

print(f(1, b=2))
try:
    f(1, 2)          # b 只能按关键字传入 → TypeError
except TypeError as e:
    print('TypeError:', e)

(1, 2)
TypeError: f() takes 1 positional argument but 2 were given


> 仅限关键字参数**不必有默认值**：像示例中的 `b` 一样，它被强制要求传入实参。

## 7.7.1 仅限位置参数

从 Python 3.8 开始，用户定义的函数**也能**指定**仅限位置参数（Positional-only Parameter）**。这个特性其实一直存在于内置函数中：比如 `divmod(a, b)` 只能用位置参数调用，写成 `divmod(a=10, b=4)` 会报错。

定义仅限位置参数的办法：在它们后面加一个 **`/`**。

In [15]:
# 示例 7.12：模拟 divmod 内置函数
def divmod(a, b, /):          # / 之前的 a、b 只能按位置传入
    return (a // b, a % b)

print(divmod(10, 4))
try:
    divmod(a=10, b=4)         # 按关键字传 → TypeError
except TypeError as e:
    print('TypeError:', e)

(2, 2)
TypeError: divmod() got some positional-only arguments passed as keyword arguments: 'a, b'


`/` 之前是仅限位置参数，`/` 之后照旧。**若签名里没有 `/`，就说明没有仅限位置参数。** （在 Python 3.7 或更早版本中，参数列表里的 `/` 是语法错误。）

把它用到 `tag` 上：若希望 `name` 只能按位置传入，就在它后面加 `/`。

In [16]:
# 示例 7.13：给 tag 的 name 加上"仅限位置"
def tag(name, /, *content, class_=None, **attrs):
    if class_ is not None:
        attrs['class'] = class_
    attr_pairs = (f' {attr}="{value}"' for attr, value in sorted(attrs.items()))
    attr_str = ''.join(attr_pairs)
    if content:
        elements = (f'<{name}{attr_str}>{c}</{name}>' for c in content)
        return '\n'.join(elements)
    else:
        return f'<{name}{attr_str} />'

print(tag('p', 'hello'))
try:
    tag(name='p')             # name 是仅限位置参数 → TypeError
except TypeError as e:
    print('TypeError:', e)

<p>hello</p>
TypeError: tag() missing 1 required positional argument: 'name'


> **记忆要点：**
> - `*` 之后 → 仅限关键字参数（**必须**按关键字传）；
> - `/` 之前 → 仅限位置参数（**只能**按位置传）；
> - 二者可以同时出现：`def f(a, /, b, *, c)`。

搞清楚了灵活的参数声明，本章余下部分介绍标准库中支持函数式编程的常用包。

# 7.8 支持函数式编程的包

虽然 Guido 明确表示 Python 不是为函数式编程设计的语言，但有了**一等函数**、模式匹配，以及 `operator` 与 `functools` 等包，函数式风格在 Python 里也能用得不错。

## 7.8.1 operator 模块

函数式编程中，把**算术运算符当作函数**会很方便。例如不用递归算阶乘：求和可以用 `sum`，可"连续求积"却没有现成函数——这时用 `reduce`，并给它一个"两项相乘"的函数。用 `lambda` 是这样：

In [17]:
# 示例 7.14：用 reduce 与 lambda 实现阶乘
from functools import reduce

def factorial(n):
    return reduce(lambda a, b: a * b, range(1, n + 1))

print(factorial(5))

120


`operator` 模块为各种运算符提供了等效函数，**无需再手写 `lambda a, b: a * b`**。用 `mul` 改写：

In [18]:
# 示例 7.15：用 reduce 与 mul 函数实现阶乘
from functools import reduce
from operator import mul

def factorial(n):
    return reduce(mul, range(1, n + 1))

print(factorial(10))

3628800


`operator` 还提供一组**工厂函数**：`itemgetter` 与 `attrgetter`，用来替代"从序列取项 / 读对象属性"的 lambda。

`itemgetter` 的常见用法：按元组中的某个字段给元组列表排序。下面的例子里，按**国家代码**（第 2 个字段）排序打印城市。

In [19]:
# 示例 7.16：用 itemgetter 排序一个元组列表
metro_data = [
    ('Tokyo', 'JP', 36.933, (35.689722, 139.691667)),
    ('Delhi NCR', 'IN', 21.935, (28.613889, 77.208889)),
    ('Mexico City', 'MX', 20.142, (19.433333, -99.133333)),
    ('New York-Newark', 'US', 20.104, (40.808611, -74.020386)),
    ('São Paulo', 'BR', 19.649, (-23.547778, -46.635833)),
]

from operator import itemgetter
for city in sorted(metro_data, key=itemgetter(1)):
    print(city)

print('---')
# 传多个索引：构建的函数返回"提取值构成的元组"，便于按多键排序
cc_name = itemgetter(1, 0)
for city in metro_data:
    print(cc_name(city))

('São Paulo', 'BR', 19.649, (-23.547778, -46.635833))
('Delhi NCR', 'IN', 21.935, (28.613889, 77.208889))
('Tokyo', 'JP', 36.933, (35.689722, 139.691667))
('Mexico City', 'MX', 20.142, (19.433333, -99.133333))
('New York-Newark', 'US', 20.104, (40.808611, -74.020386))
---
('JP', 'Tokyo')
('IN', 'Delhi NCR')
('MX', 'Mexico City')
('US', 'New York-Newark')
('BR', 'São Paulo')


> `itemgetter` 用的是 `[]` 运算符，所以**不仅支持序列，还支持映射**以及任何实现了 `__getitem__` 的类。

`attrgetter` 类似，但按**名称**提取对象属性；传多个属性名就返回元组；属性名里若含 `.`（点），它会**深入嵌套对象**去取属性。

In [20]:
# 示例 7.17：用 attrgetter 处理示例 7.16 定义的 metro_data
from collections import namedtuple
LatLon = namedtuple('LatLon', 'lat lon')
Metropolis = namedtuple('Metropolis', 'name cc pop coord')
metro_areas = [Metropolis(name, cc, pop, LatLon(lat, lon))
               for name, cc, pop, (lat, lon) in metro_data]

print(metro_areas[0])
print(metro_areas[0].coord.lat)

from operator import attrgetter
name_lat = attrgetter('name', 'coord.lat')          # 取名字 + 嵌套的纬度

for city in sorted(metro_areas, key=attrgetter('coord.lat')):
    print(name_lat(city))

Metropolis(name='Tokyo', cc='JP', pop=36.933, coord=LatLon(lat=35.689722, lon=139.691667))
35.689722
('São Paulo', -23.547778)
('Mexico City', 19.433333)
('Delhi NCR', 28.613889)
('Tokyo', 35.689722)
('New York-Newark', 40.808611)


下面是 `operator` 模块中除 `_` 开头（实现细节）以外的全部函数：

In [21]:
import operator
print([name for name in dir(operator) if not name.startswith('_')])

['abs', 'add', 'and_', 'attrgetter', 'call', 'concat', 'contains', 'countOf', 'delitem', 'eq', 'floordiv', 'ge', 'getitem', 'gt', 'iadd', 'iand', 'iconcat', 'ifloordiv', 'ilshift', 'imatmul', 'imod', 'imul', 'index', 'indexOf', 'inv', 'invert', 'ior', 'ipow', 'irshift', 'is_', 'is_not', 'isub', 'itemgetter', 'itruediv', 'ixor', 'le', 'length_hint', 'lshift', 'lt', 'matmul', 'methodcaller', 'mod', 'mul', 'ne', 'neg', 'not_', 'or_', 'pos', 'pow', 'rshift', 'setitem', 'sub', 'truediv', 'truth', 'xor']


这些名字大多一看就懂。以 **`i` 为前缀**、后面接另一个运算符的（如 `iadd`、`iand`）对应**增量赋值运算符**（`+=`、`&=` 等）：若第一个参数可变，它们**就地修改**第一个参数；否则效果与不带 `i` 的版本一样，直接返回运算结果。

`operator` 里最后要介绍的是 `methodcaller`——它动态创建的函数会在对象上**调用指定的方法**。

In [22]:
# 示例 7.18：methodcaller 示例
from operator import methodcaller
s = 'The time has come'
upcase = methodcaller('upper')
print(upcase(s))
hyphenate = methodcaller('replace', ' ', '-')      # 顺便绑定了额外参数
print(hyphenate(s))
print(str.upper(s))                                # 若只想用 str.upper，直接这么写即可

THE TIME HAS COME
The-time-has-come
THE TIME HAS COME


第二个测试说明：`methodcaller` 也能像 `functools.partial` 那样**部分应用、冻结某些参数**。这就是下一节的主题。

## 7.8.2 用 functools.partial 冻结参数

`functools.partial` 根据一个可调用对象产出**新的可调用对象**，新对象为原对象的某些参数**绑定了预定值**。它能把"接受多个参数的函数"改造成"需要更少参数的回调 API"。

In [23]:
# 示例 7.19：用 partial 进行函数改造
from operator import mul
from functools import partial
triple = partial(mul, 3)                # 把第一个位置参数绑定为 3
print(triple(7))
print(list(map(triple, range(1, 10))))  # 注意：这里 mul 不能直接和 map 一起用

21
[3, 6, 9, 12, 15, 18, 21, 24, 27]


更实用的例子是 `unicodedata.normalize`：处理多国语言文本时，常常需要先规范化再比较/排序字符串。

In [24]:
# 示例 7.20：用 partial 构建一个便利的 Unicode 规范化函数
import unicodedata, functools
nfc = functools.partial(unicodedata.normalize, 'NFC')
s1 = 'café'
s2 = 'cafe\u0301'          # e + 组合重音符
print(s1, s2)              # 看起来一样
print(s1 == s2)            # 但直接比较不等（码点序列不同）
print(nfc(s1) == nfc(s2))  # 规范化之后就相等了

café café
False
True


`partial` 的第一个参数是**可调用对象**，后面是任意数量的要绑定的**位置参数**和**关键字参数**。把它用到示例 7.9 的 `tag` 上，冻结一个位置参数和一个关键字参数：

In [25]:
# 示例 7.21：将 partial 应用到示例 7.9 的 tag 函数上
picture = partial(tag, 'img', class_='pic-frame')   # 冻结 name='img' 与 class_='pic-frame'
print(picture(src='wumpus.jpeg'))
print(picture)
print(picture.func)       # 访问原始函数
print(picture.args)       # 绑定的位置参数
print(picture.keywords)   # 绑定的关键字参数

<img class="pic-frame" src="wumpus.jpeg" />
functools.partial(<function tag at 0x000002542D6280E0>, 'img', class_='pic-frame')
<function tag at 0x000002542D6280E0>
('img',)
{'class_': 'pic-frame'}


> **记忆要点：`functools.partial` 对象提供 `.func` / `.args` / `.keywords` 三个属性**，分别访问原始函数、已绑定的位置参数、已绑定的关键字参数。

`functools.partialmethod` 功能与 `partial` 相同，但专为**方法**设计。`functools` 里还有 `cache`、`singledispatch` 等可作装饰器的高阶函数——留到第 9 章。

# 7.9 本章小结

本章探讨了 Python 函数的"**一等（First-Class）**"本性。核心思想是：

> 你可以把函数赋值给变量、传给其他函数、存进数据结构、访问它的属性——这让框架和工具能够对函数进行各种操作。

- **高阶函数**是函数式编程的主要内容，在 Python 中很常见：`sorted`、`min`、`max` 以及 `functools.partial` 都是常用的高阶函数。
- 有了**列表推导式 / 生成器表达式**以及 `sum`、`all`、`any` 等规约函数，`map`、`filter`、`reduce` 已不像从前那样常用。
- 自 Python 3.6 起共有 **9 种可调用对象**，从 `lambda` 创建的简单函数，到实现 `__call__` 的类实例；生成器与协程也是可调用对象，但行为差异很大。所有可调用对象都能用 `callable()` 检测。
- 可调用对象支持丰富的形参声明语法：**仅限关键字参数**、**仅限位置参数**、以及变量注解。
- `operator` 模块（`mul`、`add`、`itemgetter`、`attrgetter`、`methodcaller`、`i` 前缀的增量函数）与 `functools.partial`，把实现函数式编程的复杂度降到最低，也减少了对复杂 lambda 语法的需求。

# 7.10 延伸阅读

- 后续三章：第 8 章**函数中的类型提示**、第 9 章**装饰器与闭包**、第 10 章**用一等函数实现设计模式**；
- 《Python 语言参考》"3.2 The standard type hierarchy"——9 种可调用类型及所有内置类型的权威列表；
- 《Python Cookbook, 3rd ed》（Beazley & Jones）第 7 章——本书第 7、9 章的好补充；
- **PEP 3102 – Keyword-Only Arguments**（仅限关键字参数）；
- A. M. Kuchling《Python 函数编程 HOWTO》——Python 函数式编程最佳入门（偏重迭代器与生成器）；
- StackOverflow "Python: Why is functools.partial necessary?" 中 Alex Martelli 的回答；
- 作者本人的演讲《Beyond Paradigms》——探讨"Python 是函数式语言吗？"。

# 杂谈

## Python 是一门函数式语言吗？

2000 年的 Zope 研讨会上，有人问 Guido：Python 的哪些特性是从其他语言借鉴的？他答："**Python 中的所有优点都是从其他语言中窃取的。**"

布朗大学教授 Shriram Krishnamurthi 认为，编程语言的"**范式（Paradigms）**已近末日"——与其按分类法给语言贴标签，不如**把它们看作各种特征的集合**。

即使函数式编程不是 Guido 的目标，Python 的一等函数特性为函数式编程打开了大门：
- 最初把 `lambda` 加进 Python，是为了支持 `map`、`filter`、`reduce`（Amrit Prem 于 1994 年贡献给 Python 1.0）；
- 这些函数最早出现在函数式语言 **Lisp** 中。但 Lisp **不限制** lambda 内部能做什么——因为 Lisp 里一切都是表达式；而 Python 是**面向语句**的语法，表达式不能包含语句（`try/catch` 等），这正是编写 lambda 时最常遇到的限制。这是 Python 高度可读性付出的代价。
- 讽刺的是：从 Haskell 借来**列表推导式**后，反而大大减少了对 `map`、`filter` 和 `lambda` 的需求。

> 除语法限制外，Python 广泛采用函数式编程的最大障碍是**缺乏尾部调用消除（Tail-call Elimination）**。这是一项让"尾部递归"内存高效的技术。Guido 在博文《Tail Recursion Elimination》里解释了它不适合 Python 的原因——其中前 3 个（也是最重要的）都是**可用性**原因。

**结论：从设计上讲，无论函数式语言如何定义，Python 都不是函数式语言。它只是从函数式语言中借鉴了一些好想法。**

## 匿名函数的问题

除了 Python 特有的语法限制，匿名函数在任何语言中都有一个严重缺点：**它们没有名字。**
- 函数有了名字之后，**堆栈跟踪（traceback）会更好读**；
- 匿名函数是方便的捷径，但人们容易"得意忘形"——尤其是像 JavaScript 那样鼓励深层嵌套匿名函数的语言；**嵌套太深不利于调试和处理错误**；
- Python 的异步编程之所以更结构化，或许就是因为 lambda 语法受限，**防止了 lambda 的滥用**，强制采用更明确的方法。

Promise、Future、Deferred 是现代异步 API 中的概念，它们与协程一起，为所谓的"回调地狱"提供了出路——这个话题留到第 21 章。